# Loading dataset and preprocessing

In [13]:
import pandas as pd
df = pd.read_csv("Ecommerce_data.csv")
df.head(5)

,Text,label
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household
1,"Contrast living Wooden Decorative Box,Painted ...",Household
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories


In [14]:
df['label'].value_counts()

,count
label,
Household,6000
Electronics,6000
Clothing & Accessories,6000
Books,6000


In [15]:
# Mapping target column
df['label_num'] = df['label'].map({
    'Household' : 0,
    'Books': 1,
    'Electronics': 2,
    'Clothing & Accessories': 3
})

df.head(5)

,Text,label,label_num
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household,0
1,"Contrast living Wooden Decorative Box,Painted ...",Household,0
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics,2
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories,3
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories,3


In [16]:
# Preprocess function
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger", "ner"])

def preprocess(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
        results.append(" ".join(tokens))
    return results

In [17]:
df["preprocessed_txt"] = preprocess(df["Text"])

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [18]:
df.head()

,Text,label,label_num,preprocessed_txt
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household,0,urban ladder eisner low study office computer ...
1,"Contrast living Wooden Decorative Box,Painted ...",Household,0,contrast living wooden decorative box painted ...
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics,2,io crest sy pci40010 pci raid host controller ...
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories,3,isakaa baby socks born 8 years- pack 4 6 8 12 ...
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories,3,indira designer women art mysore silk saree bl...


In [ ]:
# Train test split
from sklearn.model_selection import train_test_split
X = df['preprocessed_txt']
Y = df['label_num']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=df['label_num'])

# Applying BoW and Naive Bayes, Random forest

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9608333333333333

In [ ]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1200
           1       0.98      0.91      0.94      1200
           2       0.96      0.97      0.97      1200
           3       0.97      0.99      0.98      1200

    accuracy                           0.96      4800
   macro avg       0.96      0.96      0.96      4800
weighted avg       0.96      0.96      0.96      4800



In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('rf', RandomForestClassifier())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9775

In [ ]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1200
           1       0.98      0.97      0.98      1200
           2       0.98      0.98      0.98      1200
           3       0.98      0.99      0.99      1200

    accuracy                           0.98      4800
   macro avg       0.98      0.98      0.98      4800
weighted avg       0.98      0.98      0.98      4800



# Applying n-grams and Naive Bayes

In [ ]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(2,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9727083333333333

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.97      0.96      1200
           1       0.99      0.94      0.97      1200
           2       0.97      0.98      0.97      1200
           3       0.99      0.99      0.99      1200

    accuracy                           0.97      4800
   macro avg       0.97      0.97      0.97      4800
weighted avg       0.97      0.97      0.97      4800



# Applying TF-IDF and Naive Bayes, Random forest

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9627083333333334

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1200
           1       0.99      0.92      0.95      1200
           2       0.96      0.98      0.97      1200
           3       0.97      0.99      0.98      1200

    accuracy                           0.96      4800
   macro avg       0.96      0.96      0.96      4800
weighted avg       0.96      0.96      0.96      4800



In [ ]:
clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('rf', RandomForestClassifier())
])

clf.fit(X_train, Y_train)
clf.score(X_test, Y_test)

0.9777083333333333

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1200
           1       0.98      0.97      0.98      1200
           2       0.98      0.98      0.98      1200
           3       0.99      0.99      0.99      1200

    accuracy                           0.98      4800
   macro avg       0.98      0.98      0.98      4800
weighted avg       0.98      0.98      0.98      4800



### Making predictions

In [ ]:
X_test[:3]

,preprocessed_txt
5185,meditations marcus aurelius c183
11947,boat rockerz 510 wireless bluetooth headphones...
13379,deckup lexis engineered wood 3 shelf matte fin...


In [ ]:
Y_test[:3]

,label_num
5185,1
11947,2
13379,0


In [ ]:
y_pred[:3]

array([1, 2, 0])

# Using fasttext vectors

In [19]:
df['label'] = df['label'].replace("Clothing & Accessories", "Clothing_Accessories")

In [20]:
#When you train a fasttext model, it expects labels to be specified with label prefix.
df['label'] = '__label__' + df['label'].astype(str)
df.head(5)

,Text,label,label_num,preprocessed_txt
0,Urban Ladder Eisner Low Back Study-Office Comp...,__label__Household,0,urban ladder eisner low study office computer ...
1,"Contrast living Wooden Decorative Box,Painted ...",__label__Household,0,contrast living wooden decorative box painted ...
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,__label__Electronics,2,io crest sy pci40010 pci raid host controller ...
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,__label__Clothing_Accessories,3,isakaa baby socks born 8 years- pack 4 6 8 12 ...
4,Indira Designer Women's Art Mysore Silk Saree ...,__label__Clothing_Accessories,3,indira designer women art mysore silk saree bl...


In [21]:
# Combining label and description
df['label_description'] = df['label'] + ' ' + df['preprocessed_txt']
df.head(2)

,Text,label,label_num,preprocessed_txt,label_description
0,Urban Ladder Eisner Low Back Study-Office Comp...,__label__Household,0,urban ladder eisner low study office computer ...,__label__Household urban ladder eisner low stu...
1,"Contrast living Wooden Decorative Box,Painted ...",__label__Household,0,contrast living wooden decorative box painted ...,__label__Household contrast living wooden deco...


In [22]:
# Train test split
from sklearn.model_selection import train_test_split
df = df[['label_description']]

train, test = train_test_split(df, test_size=0.2)

In [23]:
train.to_csv("ecommerce.train", columns=["label_description"], index=False, header=False)
test.to_csv("ecommerce.test", columns=["label_description"], index=False, header=False)

In [26]:
!pip install fasttext

In [28]:
import fasttext

model = fasttext.train_supervised(input="ecommerce.train")
model.test("ecommerce.test")

(4520, 0.9736725663716814, 0.9736725663716814)

First parameter is test size, 2nd and 3rd parameters are precision and recall respectively

### Making predictions

In [30]:
model.predict(["wintech assemble desktop pc cpu 500 gb sata hdd 4 gb ram intel c2d processor 3"])

([['__label__Electronics']], [array([0.99974984], dtype=float32)])

In [31]:
model.predict(["ockey men's cotton t shirt fabric details 80 cotton 20 polyester super combed cotton rich fabric"])

([['__label__Clothing_Accessories']], [array([0.99942195], dtype=float32)])

In [32]:
model.predict(["think and grow rich deluxe edition"])

([['__label__Books']], [array([1.0000031], dtype=float32)])

In [45]:
model.get_nearest_neighbors("intel")

[(0.9958312511444092, 'corporation'),
 (0.995796799659729, 'pentax'),
 (0.9957926869392395, 'she1405bk/94'),
 (0.9957926869392395, 'convinient'),
 (0.9957820177078247, 'bs179tx'),
 (0.9957451820373535, 'warping'),
 (0.9957308769226074, 'eg2'),
 (0.9956988096237183, '3gb+32'),
 (0.9956957101821899, '54.6'),
 (0.995650053024292, 'omtp')]

In [47]:
model.get_nearest_neighbors("pant")

[(0.9961593151092529, 'duke'),
 (0.9961557984352112, 'pink_free'),
 (0.9961480498313904, 'cowboy'),
 (0.996143639087677, 'jeggings'),
 (0.9961004257202148, 'underpants'),
 (0.9960947036743164, 'cufflink'),
 (0.99607253074646, 'luke'),
 (0.9960715770721436, 'sajani'),
 (0.996070384979248, 'regenerates'),
 (0.9960699081420898, 'calluses')]